# PetLens V2 — fixed 100-step T4 pilot
Upload the package ZIP and `.sha256` using the left Files panel. Select a T4 GPU runtime. Setup only checks compatibility; training runs in its own cell. Download results before disconnecting. No GitHub push is needed.


In [ ]:
from pathlib import Path
import hashlib, zipfile
archive = Path('/content/petlens-colab-100step.zip')
expected = Path(str(archive) + '.sha256').read_text().split()[0]
assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected, 'Archive SHA256 mismatch'
root = Path('/content/petlens-colab-100step')
assert not root.exists(), 'Already extracted: use existing directory or a fresh runtime'
with zipfile.ZipFile(archive) as z:
    for entry in z.infolist():
        path = Path(entry.filename)
        
        assert not path.is_absolute() and '..' not in path.parts
        assert path.parts[0] == 'petlens-colab-100step'
        assert (entry.external_attr >> 16) & 0o170000 != 0o120000
    z.extractall('/content')
%cd /content/petlens-colab-100step
!python training/rtmpose_paw_v2/colab/verify_package.py


## Setup and preflight (no optimizer steps)
Downloads the pinned software environment. Verifies package hashes, exact pretrained weights, data pipelines and real CUDA kernels/model gradients. A failure stops this cell.


In [ ]:
import subprocess
subprocess.run(['bash', 'training/rtmpose_paw_v2/colab/setup.sh'], check=True)


## Start the fixed 100-step pilot
Run this cell only when ready to train. It evaluates the baseline, performs 100 optimizer steps, saves `iter_100.pth`, evaluates the candidate and archives outputs. The diagnostic frames are from the same video and cannot establish generalization.


In [ ]:
subprocess.run(['bash', 'training/rtmpose_paw_v2/colab/run.sh', 'train'], check=True)


## Download results
The archive includes the final model, metrics, paired predictions, logs, resolved configuration, dependency versions, hashes and adapter source.


In [ ]:
from google.colab import files
files.download('/content/petlens-pilot-results.tar.gz')
